
# TMaze: use preprocessed positions + trim info to linearize trials

Pipeline:

1. Load:
   * `*_TMaze_preprocessed_positions.csv`
   * `*_TMaze_trials.csv`
   * `dlc_trim_info.csv` (for `start_frame` of trimmed miniscope)
2. Use `x_smooth`, `y_smooth` from the preprocessed positions as 2D trajectory.
3. Adjust trial timing so that frames are expressed relative to the **trimmed miniscope video**, i.e. subtract `start_frame` from `dlc_trim_info.csv`.
4. Linearize the coordinates along TMaze centerlines for each trial.
5. Make diagnostic plots per trial type.
6. Optionally save a CSV with linearized trial coordinates in the same directory as the input CSVs.


In [ ]:

import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100


In [ ]:

def _project_onto_segment(p, a, b):
    """Project point p onto segment a-b."""
    ab = b - a
    ab2 = np.dot(ab, ab)
    if ab2 == 0:
        return a, 0.0, np.sum((p - a)**2)
    t = np.dot(p - a, ab) / ab2
    t_clipped = np.clip(t, 0.0, 1.0)
    proj = a + t_clipped * ab
    dist2 = np.sum((p - proj)**2)
    return proj, t_clipped, dist2

def load_ports_from_metadata(metadata_csv_path):
    meta = pd.read_csv(metadata_csv_path)
    row = meta.loc[meta["Unnamed: 0"] == "port_rois"]
    port_str = row["value"].iloc[0]

    # e.g. "{'base': (880.0, 900.0, 50.0), 'left': (...), 'right': (...)}"
    ports_raw = ast.literal_eval(port_str)

    ports = {}
    for key, val in ports_raw.items():
        x, y = float(val[0]), float(val[1])   # ignore radius
        ports[key] = np.array([x, y], dtype=float)
    return ports



def linearize_positions_along_polyline(location, poly_points):
    """
    location    : (N, 2) array of [x, y] positions
    poly_points : (M, 2) array of [x, y] defining the maze centerline

    Returns:
    linear_pos  : (N,) array, 1D position (same units as coords)
    """
    loc = np.asarray(location, float)
    poly = np.asarray(poly_points, float)

    seg_vecs = poly[1:] - poly[:-1]
    seg_len = np.linalg.norm(seg_vecs, axis=1)
    cum_len = np.concatenate([[0.0], np.cumsum(seg_len)])

    N = loc.shape[0]
    linear_pos = np.full(N, np.nan)

    for i in range(N):
        p = loc[i]
        if np.any(np.isnan(p)):
            continue

        best_dist2 = np.inf
        best_s = np.nan

        for s_idx in range(len(seg_len)):
            a = poly[s_idx]
            b = poly[s_idx + 1]
            _, t, d2 = _project_onto_segment(p, a, b)
            if d2 < best_dist2:
                best_dist2 = d2
                best_s = cum_len[s_idx] + t * seg_len[s_idx]

        linear_pos[i] = best_s

    return linear_pos


# --- TMaze geometry (adapt if needed) ---
base, left, right, poly_BL, poly_BR, poly_LR = make_polylines_from_ports(ports)
# poly_BL = [base -> left], poly_BR = [base -> right], poly_LR = [left -> right]


bl_types = {"base_to_left", "left_to_base"}
br_types = {"base_to_right", "right_to_base"}
lr_types = {"left_to_right", "right_to_left"}


def get_linpos_for_trial_type(trial_type, linpos_BL, linpos_BR, linpos_LR):
    if trial_type in bl_types:
        return linpos_BL
    elif trial_type in br_types:
        return linpos_BR
    elif trial_type in lr_types:
        return linpos_LR
    else:
        raise ValueError(f"Unknown trial_type: {trial_type}")


In [ ]:

def load_trim_info(trim_csv_path):
    """Load dlc_trim_info.csv and return a Series with at least start_frame."""
    trim_df = pd.read_csv(trim_csv_path)
    if trim_df.empty:
        raise ValueError("dlc_trim_info.csv is empty.")
    row = trim_df.iloc[0]
    if "start_frame" not in trim_df.columns:
        raise ValueError("dlc_trim_info.csv must contain a 'start_frame' column.")
    return row


def load_trials(trials_csv_path):
    """Load TMaze trials CSV; expects start_frame, end_frame, trial_type."""
    trials = pd.read_csv(trials_csv_path)
    required = {"start_frame", "end_frame", "trial_type"}
    missing = required - set(trials.columns)
    if missing:
        raise ValueError(f"Trials CSV is missing columns: {missing}")
    return trials


def load_preprocessed_positions(preproc_csv_path,
                                x_col="x_smooth",
                                y_col="y_smooth"):
    """
    Load TMaze_preprocessed_positions.csv and return 2D trajectory as (N, 2).

    By default uses x_smooth, y_smooth, but you can switch to x_interp, y_interp.
    """
    df = pd.read_csv(preproc_csv_path)
    for c in ["frame", x_col, y_col]:
        if c not in df.columns:
            raise ValueError(f"Preprocessed positions CSV missing column: {c}")
    coords = df[[x_col, y_col]].to_numpy(dtype=float)
    frames = df["frame"].to_numpy(dtype=int)
    return coords, frames, df


In [ ]:

def adjust_trials_for_trim(trials, trim_start_frame, n_frames_trimmed=None):
    """
    Re-express trial start/end frames relative to trimmed miniscope start.

    New columns:
        - trim_start
        - trim_end
    """
    trials_adj = trials.copy()
    trim_start_frame = int(trim_start_frame)

    trials_adj["trim_start"] = trials_adj["start_frame"] - trim_start_frame
    trials_adj["trim_end"]   = trials_adj["end_frame"]   - trim_start_frame

    if n_frames_trimmed is not None:
        trials_adj["trim_start"] = trials_adj["trim_start"].clip(0, n_frames_trimmed - 1)
        trials_adj["trim_end"]   = trials_adj["trim_end"].clip(0, n_frames_trimmed - 1)

    return trials_adj


def build_trimmed_location_from_preprocessed(coords_full,
                                             frames_full,
                                             trim_start_frame):
    """
    Given full preprocessed coordinates and their frame numbers in original
    DLC space, slice to trimmed coordinates (frame 0 = trim_start_frame).

    Returns
    -------
    loc_trim : (N_trim, 2) array
    """
    trim_start_frame = int(trim_start_frame)

    if trim_start_frame < frames_full.min() or trim_start_frame > frames_full.max():
        raise ValueError("trim_start_frame is outside the range of preprocessed frames.")

    # Indices where frame >= trim_start_frame
    mask = frames_full >= trim_start_frame
    loc_trim = coords_full[mask]

    return loc_trim


In [ ]:

def linearize_trials(location_trim,
                     trials_adj,
                     save=False,
                     out_dir=None,
                     out_basename="TMaze_linearized_trials",
                     max_trials_per_type_for_plots=3):
    """
    Linearize TMaze coordinates for each trial and optionally save results.

    Parameters
    ----------
    location_trim : (N_trim, 2)
        2D trajectory in trimmed frame coordinates (index 0 == trim start).
    trials_adj : DataFrame
        Must contain trim_start, trim_end, trial_type.
    save : bool
        If True, writes a CSV to out_dir.
    out_dir : str or None
        Directory to write the CSV. If None and save=True, uses current directory.
    out_basename : str
        Base name of output CSV.
    max_trials_per_type_for_plots : int
        Number of example trials per type to display.

    Returns
    -------
    lin_df : DataFrame
        Columns: trial_index, trial_type, frame_in_trial, frame_global_trim, linpos.
    linpos_BL, linpos_BR, linpos_LR : np.ndarray
    """
    loc = np.asarray(location_trim, float)
    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location_trim must be (N_trim, 2)")

    # Full-series linearizations
    linpos_BL = linearize_positions_along_polyline(loc, poly_BL)
    linpos_BR = linearize_positions_along_polyline(loc, poly_BR)
    linpos_LR = linearize_positions_along_polyline(loc, poly_LR)

    rows = []
    for idx, tr in trials_adj.iterrows():
        ttype = tr["trial_type"]
        start = int(tr["trim_start"])
        end   = int(tr["trim_end"])

        if start < 0 or end >= loc.shape[0] or end <= start:
            continue

        lin_full = get_linpos_for_trial_type(ttype, linpos_BL, linpos_BR, linpos_LR)
        seg = lin_full[start:end+1]

        for k, val in enumerate(seg):
            rows.append({
                "trial_index": idx,
                "trial_type": ttype,
                "frame_in_trial": k,
                "frame_global_trim": start + k,
                "linpos": float(val),
            })

    lin_df = pd.DataFrame(rows)

    # --- Diagnostic plots ---
    unique_types = trials_adj["trial_type"].unique()
    for ttype in unique_types:
        tr_idxs = trials_adj.index[trials_adj["trial_type"] == ttype].tolist()
        if not tr_idxs:
            continue
        tr_idxs = tr_idxs[:max_trials_per_type_for_plots]

        for tid in tr_idxs:
            tr = trials_adj.loc[tid]
            start = int(tr["trim_start"])
            end   = int(tr["trim_end"])
            if start < 0 or end >= loc.shape[0] or end <= start:
                continue

            coords = loc[start:end+1]
            seg_lin = lin_df[lin_df["trial_index"] == tid]["linpos"].to_numpy()

            fig, axes = plt.subplots(1, 2, figsize=(10, 4))

            # 2D path
            axes[0].plot(loc[:, 0], loc[:, 1], alpha=0.2, label="all (trim)")
            axes[0].plot(coords[:, 0], coords[:, 1], "-o", label=f"trial {tid}")
            axes[0].scatter(
                [base[0], choice[0], left_end[0], right_end[0]],
                [base[1], choice[1], left_end[1], right_end[1]],
                marker="x", s=50, label="landmarks"
            )
            axes[0].set_title(f"{ttype} – trial {tid} path (trim coords)")
            axes[0].set_xlabel("x")
            axes[0].set_ylabel("y")
            axes[0].legend()

            # Linearized trace
            axes[1].plot(seg_lin, "-o")
            axes[1].set_title(f"{ttype} – trial {tid} linearized")
            axes[1].set_xlabel("frame in trial")
            axes[1].set_ylabel("linear position (a.u.)")
            axes[1].grid(True)

            plt.tight_layout()
            plt.show()

    # --- Optional save ---
    if save:
        if out_dir is None:
            out_dir = os.getcwd()
        os.makedirs(out_dir, exist_ok=True)
        out_csv = os.path.join(out_dir, out_basename + ".csv")
        lin_df.to_csv(out_csv, index=False)
        print("Saved linearized trial coordinates to:", out_csv)

    return lin_df, linpos_BL, linpos_BR, linpos_LR


In [ ]:
def make_bins(linpos, n_bins=100):
    max_val = np.nanmax(linpos)
    edges = np.linspace(0.0, max_val, n_bins + 1)
    b = np.digitize(linpos, edges, right=False) - 1   # 0..n_bins-1
    b = np.clip(b, 0, n_bins - 1)
    return edges, b

In [ ]:

# === Example usage for one session ===
# Edit these paths for your session:

base_dir         = r"/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-10T17_53_09/"  # directory that contains the CSVs
trim_info_path   = os.path.join(base_dir, "dlc_trim_info.csv")
trials_csv_path  = os.path.join(base_dir,"dlc", "2024-06-10_17-53-09_TMaze_trials.csv")
preproc_csv_path = os.path.join(base_dir,"dlc", "2024-06-10_17-53-09_TMaze_preprocessed_positions.csv")

trim_info = load_trim_info(trim_info_path)
trials    = load_trials(trials_csv_path)
coords_full, frames_full, pre_df = load_preprocessed_positions(preproc_csv_path,
                                                                x_col="x_smooth",
                                                                y_col="y_smooth")

loc_trim = build_trimmed_location_from_preprocessed(
    coords_full,
    frames_full,
    trim_start_frame=trim_info["start_frame"],
)

trials_adj = adjust_trials_for_trim(
    trials,
    trim_start_frame=trim_info["start_frame"],
    n_frames_trimmed=loc_trim.shape[0],
)

lin_df, linpos_BL, linpos_BR, linpos_LR = linearize_trials(
    loc_trim,
    trials_adj,
    save=False,
    out_dir=base_dir,
    out_basename="TMaze_linearized_trials"
)
